In [ ]:
import numpy as np

In [ ]:
def f_2(val):
    X = val[0]
    Y = val[1]
    r2 = X**2 + Y**2
    z = (r2)**0.25 * (np.sin(50 * (r2)**0.1)**2 + 1)
    return z

def fitness(x):
    return -f_2(x)



In [ ]:
class SSO():
    def __init__(self,fitness_function, bounds =None, problem_type="continuous", refset_size = 10, pop_size = 100):
        
        self.fitness = fitness_function
        self.problem_type = problem_type
        self.refset_size = refset_size
        self.pop_size = pop_size

        if problem_type == "continuous":
            self.bounds = bounds
            self._generate_diverse_solution = self._generate_diverse_solution_cont
            self._generate_neighbors = self._generate_neighbors_cont
            self._solution_combination = self._solution_combination_cont
            self._select_most_diverse = self._select_most_diverse_cont

    def run(self, max_iters):
        P = self._diversification_generation() #generamos la poblacion inicial
        P = [self._improvement(sol) for sol in P] #potenciamos cada una de las soluciones iniciales

        RefSet = self._reference_set_update_init(P) #creamos el conjutno de referencia
        nuevas = RefSet.copy() # iniaiclamente las combino con si mismo
    
        for _ in range(max_iters):
            #generamos el subconjunto
            subsets = self._subset_generation(RefSet, nuevas) #hace la combinatoria de Referencia y nuevas
            nuevas = []

            #recorremos todas las combinaciones de soluciones
            for subset in subsets:
                sol = self._solution_combination(subset) #combino analiticamente las solucioens
                sol = self._improvement(sol) #la exploto localmente
                fue_agregada = self._reference_set_update(RefSet, sol) #se agrega si es mejor que la peor de refset
                if fue_agregada==True:
                    nuevas.append(sol)

            if len(nuevas)==0: #ninguna combinacion es mejor
                break

        return self._best_solution(RefSet) #retorno la mejor solucion de este conjunto
    
    def _diversification_generation(self):
        #La poblacion inicial es mitad por fitnes mitad por aleatoriedad
        P=[]

        while len(P) < self.pop_size:
            sol = self._generate_diverse_solution() #dependiendo de la codificacion que estemos trabajando
            P.append(sol)

        return P
    
    def _improvement(self, sol):
        improved = True

        while improved == True:
            improved = False
            vecinos = self._generate_neighbors(sol) #generamos los vecinos

            for vecino in vecinos:
                if self.fitness(vecino)> self.fitness(sol): #si el vecino tiene mejor fitnes
                    sol = vecino
                    improved = True
                    break #si encontro un mejor vecino que ya corte nomas
        
        return sol
    
    def _reference_set_update_init(self, P):
        P = sorted(P,key = self.fitness, reverse = True)

        mejores = P[:self.refset_size // 2]
        restantes = P[self.refset_size // 2:] #de aca muestreo aleatoriamente si es continuo

        diversas = self._select_most_diverse(
            restantes,
            self.refset_size // 2,
            mejores
        )

        return mejores + diversas
    
    def _reference_set_update(self, RefSet, nueva_sol):
        peor = min(RefSet, key = self.fitness)

        if self.fitness(nueva_sol)>self.fitness(peor):

            idx = next(
                i for i, s in enumerate(RefSet)
                if np.array_equal(s, peor)
            )

            RefSet.pop(idx)
            RefSet.append(nueva_sol)

            return True

        return False
    
    def _subset_generation(self, RefSet, nuevas):
        subsets = []

        for nueva in nuevas:
            for sol in RefSet:

                if np.array_equal(sol, nueva): #evita combinar las mismas soluciones
                    continue

                subsets.append((nueva, sol))

        return subsets
    

    # Caso continuo
    def _generate_diverse_solution_cont(self):
        sol = np.array([
            np.random.uniform(low, high)
            for low, high in self.bounds
        ])

        return sol
    
    def _generate_neighbors_cont(self, sol, n_neighbors = 20, sigma = 0.1):
        vecinos = []

        for _ in range(n_neighbors):

            vecino = sol.copy()

            for i, (low, high) in enumerate(self.bounds):
                rango = high - low

                vecino[i] += np.random.normal(0,sigma * rango)
                
                vecino[i] = np.clip(vecino[i],low,high)

            vecinos.append(vecino)

        return vecinos
    
    def _solution_combination_cont(self, subset):
        #combinacion convexa
        x1, x2 = subset

        alpha = np.random.rand()

        hijo = (alpha * x1 + (1 - alpha) * x2)

        return hijo

    def _select_most_diverse_cont(self,candidatos,k,seleccionadas):
        #usando el conjunto de referencia, usamos las que mas lejos en norma estan

        candidatos = candidatos.copy()

        diversas = []

        while len(diversas) < k and len(candidatos) > 0:

            mejor_candidato = None
            mejor_distancia = -np.inf

            for candidato in candidatos:

                conjunto = seleccionadas + diversas

                distancia_min = min(
                    np.linalg.norm(candidato - s)
                    for s in conjunto
                )

                if distancia_min > mejor_distancia:
                    mejor_distancia = distancia_min
                    mejor_candidato = candidato

            diversas.append(mejor_candidato)

            idx = next(
                i for i, c in enumerate(candidatos)
                if np.array_equal(c, mejor_candidato)
            )

            candidatos.pop(idx)

        return diversas

    def _best_solution(self, RefSet):
        return max(
            RefSet,
            key=self.fitness
        )

In [ ]:
ss = SSO(
    fitness_function=fitness,
    problem_type="continuous",
    bounds=[
        (-5, 5),
        (-5, 5)
    ],
    pop_size=100,
    refset_size=10
)

best = ss.run(max_iters=100)

print("Mejor solución:", best)
print("Valor de la función:", f_2(best))

Mejor solución: [-2.78558403e-42 -5.74081642e-43]
Valor de la función: 1.6864534354698916e-21
